In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


In [ ]:
# Load raw sample data
raw_path = PROJECT_ROOT / "data" / "raw" / "sample_taq.csv"

if not raw_path.exists():
    from mdq.generate_sample_data import build_sample_taq
    
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    build_sample_taq(days=5).to_csv(raw_path, index=False)

raw = pd.read_csv(raw_path)
raw.head(10)

In [ ]:
# Import pipeline functions
from mdq.pipeline import (
    load_raw_market_data,
    normalize_market_data,
    clean_for_research,
    write_outputs,
    run_pipeline,
)

from mdq.quality_checks import run_quality_checks

In [ ]:
raw = load_raw_market_data(raw_path)
normalized = normalize_market_data(raw)

normalized.head()

In [ ]:
problems = run_quality_checks(normalized)

for problem in problems:
    print(f"{problem.check_name} | {problem.severity} | {problem.row_count} rows")
    print(problem.message)
    print()

In [ ]:
problems_df = pd.DataFrame([
    {
        "check_name": problem.check_name,
        "severity": problem.severity,
        "row_count": problem.row_count,
        "message": problem.message,
    }
    for problem in problems
])

problems_df

In [ ]:
clean = clean_for_research(normalized)

output_dir = PROJECT_ROOT / "data" / "processed"
report_dir = PROJECT_ROOT / "reports"

write_outputs(clean, problems, output_dir, report_dir)

clean.head()

In [ ]:
report_path = PROJECT_ROOT / "reports" / "data_quality_report.md"

print(report_path.read_text())